In [1]:
import pandas as pd
india_change = pd.read_csv(
    r"../data/raw/India_Change.csv"
)

print("Shape:", india_change.shape)
print("\nColumns:")
print(india_change.columns.tolist())

display(india_change.head())

Shape: (66976, 10)

Columns:
['State', 'ST_CEN_CD', 'District Name', 'DISTRICT', 'DT_CEN_CD', 'Category', 'Indicator', 'NFHS 5', 'NFHS 4', 'Change']


C:\Users\ASUS\AppData\Local\Temp\ipykernel_22652\2087987504.py:2: DtypeWarning: Columns (0: DT_CEN_CD) have mixed types. Specify dtype option on import or set low_memory=False.
  india_change = pd.read_csv(


,State,ST_CEN_CD,District Name,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4,Change
0,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-19 years who are anaemic (%),48.0,66.2,18.2
1,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,All women age 15-49 years who are anaemic (%),38.3,55.2,16.9
2,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,Children age 6-59 months who are anaemic,37.7,50.5,12.8
3,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,Nonpregnant women age 15-49 years who are anae...,38.4,54.5,16.1
4,Andaman & Nicobar Island,35,Nicobar,Nicobar_1,1.0,Anaemia among Children and Women,Pregnant women age 15-49 years who are anaemic,0.0,0.0,0.0


In [2]:
# Data types
print("DATA TYPES")
display(india_change.dtypes)

# Missing values
print("\nMISSING VALUES")
missing_audit = pd.DataFrame({
    'Missing_Count': india_change.isnull().sum(),
    'Missing_Percentage': (
        india_change.isnull().mean() * 100
    )
})
display(missing_audit)

# Duplicate records
print("\nDUPLICATE AUDIT")
print(
    "Exact duplicate rows:",
    india_change.duplicated().sum()
)

print(
    "Duplicate district-indicator combinations:",
    india_change.duplicated(
        subset=['State', 'District Name', 'Indicator']
    ).sum()
)

# Verify the Change formula
print("\nCHANGE FORMULA AUDIT")

formula_difference = (
    india_change['NFHS 4'] - india_change['NFHS 5']
)

formula_match = (
    india_change['Change'].round(6)
    == formula_difference.round(6)
)

print("Rows following NFHS 4 - NFHS 5:", formula_match.sum())
print("Rows not following this formula:", (~formula_match).sum())

# Inspect mismatches
display(
    india_change.loc[
        ~formula_match,
        ['State', 'District Name', 'Indicator',
         'NFHS 5', 'NFHS 4', 'Change']
    ].head(15)
)

DATA TYPES


State                str
ST_CEN_CD          int64
District Name        str
DISTRICT             str
DT_CEN_CD         object
Category             str
Indicator            str
NFHS 5           float64
NFHS 4           float64
Change           float64
dtype: object


MISSING VALUES


,Missing_Count,Missing_Percentage
State,0,0.000000
ST_CEN_CD,0,0.000000
District Name,0,0.000000
DISTRICT,0,0.000000
DT_CEN_CD,2,0.002986
Category,0,0.000000
Indicator,0,0.000000
NFHS 5,0,0.000000
NFHS 4,0,0.000000
Change,0,0.000000



DUPLICATE AUDIT
Exact duplicate rows: 2
Duplicate district-indicator combinations: 106

CHANGE FORMULA AUDIT
Rows following NFHS 4 - NFHS 5: 16504
Rows not following this formula: 50472


,State,District Name,Indicator,NFHS 5,NFHS 4,Change
5,Andaman & Nicobar Island,Nicobar,Male Blood sugar level high or very high (>14...,15.4,0.0,15.4
6,Andaman & Nicobar Island,Nicobar,Male Blood sugar level very high (>160 mg/dl)...,4.4,0.0,4.4
7,Andaman & Nicobar Island,Nicobar,MaleBlood sugar level high (141-160 mg/dl) (%),9.6,0.0,9.6
8,Andaman & Nicobar Island,Nicobar,Female Blood sugar level high (141-160 mg/dl)...,7.4,0.0,7.4
9,Andaman & Nicobar Island,Nicobar,Female Blood sugar level high or very high (>...,13.1,0.0,13.1
10,Andaman & Nicobar Island,Nicobar,Female Blood sugar level very high (>160 mg/d...,3.9,0.0,3.9
11,Andaman & Nicobar Island,Nicobar,Women who are literate (%),87.5,0.0,87.5
12,Andaman & Nicobar Island,Nicobar,Women with 10 or more years of schooling (%),53.5,43.7,9.8
13,Andaman & Nicobar Island,Nicobar,Breastfeeding children age 6-23 months receivi...,19.4,10.2,9.2
20,Andaman & Nicobar Island,Nicobar,Children under age 3 years breastfed within on...,55.4,34.9,20.5


In [3]:
# A. Identify how Change is calculated
india_change['nfhs5_minus_nfhs4'] = (
    india_change['NFHS 5'] - india_change['NFHS 4']
)

india_change['nfhs4_minus_nfhs5'] = (
    india_change['NFHS 4'] - india_change['NFHS 5']
)

india_change['formula_pattern'] = 'Other / neither'

india_change.loc[
    india_change['Change'].round(6) ==
    india_change['nfhs5_minus_nfhs4'].round(6),
    'formula_pattern'
] = 'NFHS 5 - NFHS 4'

india_change.loc[
    india_change['Change'].round(6) ==
    india_change['nfhs4_minus_nfhs5'].round(6),
    'formula_pattern'
] = 'NFHS 4 - NFHS 5'

# A zero baseline may be a separate convention
india_change.loc[
    (india_change['NFHS 4'] == 0) &
    (india_change['NFHS 5'] == india_change['Change']),
    'formula_pattern'
] = 'NFHS 4 is zero; Change equals NFHS 5'

formula_summary = (
    india_change.groupby(['Indicator', 'formula_pattern'])
    .size()
    .unstack(fill_value=0)
)

print("FORMULA PATTERNS BY INDICATOR")
display(formula_summary)

# B. Inspect exact duplicate rows
print("\nEXACT DUPLICATE ROWS")

exact_duplicates = india_change[
    india_change.duplicated(keep=False)
].sort_values(['State', 'District Name', 'Indicator'])

display(exact_duplicates)

# C. Inspect repeated district-indicator keys
print("\nREPEATED DISTRICT-INDICATOR COMBINATIONS")

duplicate_keys = india_change[
    india_change.duplicated(
        subset=['State', 'District Name', 'Indicator'],
        keep=False
    )
].sort_values(['State', 'District Name', 'Indicator'])

display(
    duplicate_keys[
        [
            'State', 'District Name', 'DISTRICT',
            'DT_CEN_CD', 'Indicator',
            'NFHS 5', 'NFHS 4', 'Change'
        ]
    ]
)

FORMULA PATTERNS BY INDICATOR


formula_pattern,NFHS 4 - NFHS 5,NFHS 4 is zero; Change equals NFHS 5,NFHS 5 - NFHS 4,Other / neither
Indicator,,,,
All women age 15-19 years who are anaemic (%),640,1,0,1
All women age 15-49 years who are anaemic (%),642,1,0,1
Any method (%),0,58,585,1
Any modern method (%),0,58,585,1
Average out-of-pocket expenditure per delivery in a public health facility (Rs),0,2,0,642
...,...,...,...,...
Women who are literate (%),0,633,10,1
Women who are overweight or obese,641,1,1,1
Women who have high risk waisttohip ratio,641,2,0,1



EXACT DUPLICATE ROWS


,State,ST_CEN_CD,District Name,DISTRICT,DT_CEN_CD,Category,Indicator,NFHS 5,NFHS 4,Change,nfhs5_minus_nfhs4,nfhs4_minus_nfhs5,formula_pattern
38269,Meghalaya,17,East Jaintia Hills,Jaintia Hills_7,7.0,Treatment of Childhood Diseases,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8,4.8,-4.8,NFHS 4 - NFHS 5
38270,Meghalaya,17,East Jaintia Hills,Jaintia Hills_7,7.0,Treatment of Childhood Diseases,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8,4.8,-4.8,NFHS 4 - NFHS 5
38789,Meghalaya,17,West Jaintia Hills,Jaintia Hills_7,7.0,Treatment of Childhood Diseases,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8,4.8,-4.8,NFHS 4 - NFHS 5
38790,Meghalaya,17,West Jaintia Hills,Jaintia Hills_7,7.0,Treatment of Childhood Diseases,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8,4.8,-4.8,NFHS 4 - NFHS 5



REPEATED DISTRICT-INDICATOR COMBINATIONS


,State,District Name,DISTRICT,DT_CEN_CD,Indicator,NFHS 5,NFHS 4,Change
23088,Jharkhand,Sahibganj,Sahibganj_7,7.0,All women age 15-19 years who are anaemic (%),67.6,64.2,-3.4
23089,Jharkhand,Sahibganj,Saraikela-kharsawan_24,7.0,All women age 15-19 years who are anaemic (%),74.9,88.1,13.2
23090,Jharkhand,Sahibganj,Sahibganj_7,7.0,All women age 15-49 years who are anaemic (%),71.5,61.9,-9.6
23091,Jharkhand,Sahibganj,Saraikela-kharsawan_24,7.0,All women age 15-49 years who are anaemic (%),70.2,78.8,8.6
23158,Jharkhand,Sahibganj,Sahibganj_7,7.0,Any method (%),50.4,34.6,15.8
...,...,...,...,...,...,...,...,...
23113,Jharkhand,Sahibganj,Saraikela-kharsawan_24,7.0,Women with 10 or more years of schooling (%),36.0,32.9,3.1
38269,Meghalaya,East Jaintia Hills,Jaintia Hills_7,7.0,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8
38270,Meghalaya,East Jaintia Hills,Jaintia Hills_7,7.0,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8
38789,Meghalaya,West Jaintia Hills,Jaintia Hills_7,7.0,Prevalence of symptoms of acute respiratory in...,4.8,0.0,-4.8


In [4]:
# Inspect all rows where neither simple difference matches Change

other_rows = india_change[
    india_change['formula_pattern'] == 'Other / neither'
].copy()

print("Rows with unexplained Change:", len(other_rows))

display(
    other_rows[
        [
            'State',
            'District Name',
            'Indicator',
            'NFHS 5',
            'NFHS 4',
            'Change',
            'nfhs5_minus_nfhs4',
            'nfhs4_minus_nfhs5'
        ]
    ].head(30)
)

Rows with unexplained Change: 2016


,State,District Name,Indicator,NFHS 5,NFHS 4,Change,nfhs5_minus_nfhs4,nfhs4_minus_nfhs5
60,Andaman & Nicobar Island,Nicobar,Average out-of-pocket expenditure per delivery...,2278.0,1787.0,27.5,491.0,-491.0
84,Andaman & Nicobar Island,Nicobar,Sex ratio at birth for children born in the la...,927.0,1060.0,-12.5,-133.0,133.0
85,Andaman & Nicobar Island,Nicobar,Sex ratio of the total population (females per...,973.0,957.0,1.7,16.0,-16.0
164,Andaman & Nicobar Island,North & Middle Andaman,Average out-of-pocket expenditure per delivery...,1904.0,1105.0,72.3,799.0,-799.0
188,Andaman & Nicobar Island,North & Middle Andaman,Sex ratio at birth for children born in the la...,844.0,975.0,-13.4,-131.0,131.0
189,Andaman & Nicobar Island,North & Middle Andaman,Sex ratio of the total population (females per...,950.0,951.0,-0.1,-1.0,1.0
268,Andaman & Nicobar Island,South Andaman,Average out-of-pocket expenditure per delivery...,3460.0,1298.0,166.6,2162.0,-2162.0
292,Andaman & Nicobar Island,South Andaman,Sex ratio at birth for children born in the la...,935.0,794.0,17.8,141.0,-141.0
293,Andaman & Nicobar Island,South Andaman,Sex ratio of the total population (females per...,967.0,989.0,-2.2,-22.0,22.0
372,Andhra Pradesh,Anantapur,Average out-of-pocket expenditure per delivery...,2682.0,3601.0,-25.5,-919.0,919.0


In [7]:
# Calculate percentage change
import numpy as np
india_change['calculated_pct_change'] = (
    (india_change['NFHS 5'] - india_change['NFHS 4'])
    / india_change['NFHS 4'].replace(0, np.nan)
) * 100

# Compare against the stored Change column
india_change['change_matches_pct'] = np.isclose(
    india_change['Change'],
    india_change['calculated_pct_change'],
    atol=0.11,
    equal_nan=False
)

# Summary
print("Percentage-change matches:",
      india_change['change_matches_pct'].sum())

print("Percentage-change non-matches:",
      (~india_change['change_matches_pct']).sum())

# Check percentage-change matches by indicator
pct_summary = (
    india_change.groupby('Indicator')['change_matches_pct']
    .agg(['sum', 'count'])
    .reset_index()
    .rename(columns={
        'sum': 'Percentage_Change_Matches',
        'count': 'Total_Records'
    })
)

display(pct_summary)

Percentage-change matches: 4536
Percentage-change non-matches: 62440


,Indicator,Percentage_Change_Matches,Total_Records
0,All women age 15-19 years who are anaemic (%),2,642
1,All women age 15-49 years who are anaemic (%),2,644
2,Any method (%),2,644
3,Any modern method (%),2,644
4,Average out-of-pocket expenditure per delivery...,583,644
...,...,...,...
100,Women who are literate (%),0,644
101,Women who are overweight or obese,2,644
102,Women who have high risk waisttohip ratio,0,644
103,Women whose Body Mass Index (BMI) is below nor...,3,644


In [8]:
# Rank indicators by percentage-change matches
pct_indicator_audit = (
    india_change.groupby('Indicator')
    .agg(
        total_records=('Change', 'size'),
        percentage_matches=('change_matches_pct', 'sum')
    )
    .reset_index()
)

pct_indicator_audit['match_percentage'] = (
    pct_indicator_audit['percentage_matches']
    / pct_indicator_audit['total_records'] * 100
)

display(
    pct_indicator_audit.sort_values(
        'percentage_matches',
        ascending=False
    ).head(20)
)

,Indicator,total_records,percentage_matches,match_percentage
90,Sex ratio at birth for children born in the la...,644,586,90.993789
91,Sex ratio of the total population (females per...,644,586,90.993789
4,Average out-of-pocket expenditure per delivery...,644,583,90.527950
57,Households using iodized salt (%),644,340,52.795031
84,Population living in households with an improv...,644,317,49.223602
85,Population living in households with electrici...,644,275,42.701863
20,Children age 12-23 months who received most of...,644,274,42.546584
17,Children age 12-23 months who have received BC...,644,248,38.509317
33,Children under age 5 years whose birth was reg...,644,184,28.571429
89,Registered pregnancies for which the mother re...,644,170,26.397516


In [9]:
# Select the three indicators with the highest percentage-change matches
top_indicators = [
    'Sex ratio at birth for children born in the last five years',
    'Sex ratio of the total population (females per 1,000 males)',
]

# Use indicator names directly from the audit table to avoid
# errors caused by truncated or slightly different names.
top_indicators = pct_indicator_audit.nlargest(
    3, 'match_percentage'
)['Indicator'].tolist()

for indicator in top_indicators:

    subset = india_change[
        india_change['Indicator'] == indicator
    ].copy()

    unmatched = subset[
        ~subset['change_matches_pct']
    ]

    print("\nINDICATOR:", indicator)
    print("Total records:", len(subset))
    print("Unmatched records:", len(unmatched))
    print("NFHS-4 zero records:", (unmatched['NFHS 4'] == 0).sum())

    display(
        unmatched[
            [
                'State',
                'District Name',
                'NFHS 5',
                'NFHS 4',
                'Change',
                'calculated_pct_change'
            ]
        ].head(15)
    )


INDICATOR: Sex ratio at birth for children born in the last five years (females per 1,000 males)
Total records: 644
Unmatched records: 58
NFHS-4 zero records: 58


,State,District Name,NFHS 5,NFHS 4,Change,calculated_pct_change
2164,Arunachal Pradesh,East Siang,1162.0,0.0,0.0,NaN
2268,Arunachal Pradesh,Kurung Kumey,869.0,0.0,0.0,NaN
2372,Arunachal Pradesh,Lohit,839.0,0.0,0.0,NaN
2892,Arunachal Pradesh,Tirap,865.0,0.0,0.0,NaN
3308,Arunachal Pradesh,West Siang,1060.0,0.0,0.0,NaN
4140,Assam,Dhubri,914.0,0.0,0.0,NaN
4764,Assam,Jorhat,833.0,0.0,0.0,NaN
5076,Assam,Karbi Anglong,882.0,0.0,0.0,NaN
5596,Assam,Nagaon,969.0,0.0,0.0,NaN
5804,Assam,Sivasagar,915.0,0.0,0.0,NaN



INDICATOR: Sex ratio of the total population (females per 1,000 males)
Total records: 644
Unmatched records: 58
NFHS-4 zero records: 58


,State,District Name,NFHS 5,NFHS 4,Change,calculated_pct_change
2165,Arunachal Pradesh,East Siang,1059.0,0.0,0.0,NaN
2269,Arunachal Pradesh,Kurung Kumey,997.0,0.0,0.0,NaN
2373,Arunachal Pradesh,Lohit,1001.0,0.0,0.0,NaN
2893,Arunachal Pradesh,Tirap,964.0,0.0,0.0,NaN
3309,Arunachal Pradesh,West Siang,1018.0,0.0,0.0,NaN
4141,Assam,Dhubri,1000.0,0.0,0.0,NaN
4765,Assam,Jorhat,1009.0,0.0,0.0,NaN
5077,Assam,Karbi Anglong,994.0,0.0,0.0,NaN
5597,Assam,Nagaon,1050.0,0.0,0.0,NaN
5805,Assam,Sivasagar,989.0,0.0,0.0,NaN



INDICATOR: Average out-of-pocket expenditure per delivery in a public health facility (Rs)
Total records: 644
Unmatched records: 61
NFHS-4 zero records: 61


,State,District Name,NFHS 5,NFHS 4,Change,calculated_pct_change
996,Andhra Pradesh,Prakasam,0.0,0.0,0.0,NaN
2140,Arunachal Pradesh,East Siang,12217.0,0.0,0.0,NaN
2244,Arunachal Pradesh,Kurung Kumey,9838.0,0.0,0.0,NaN
2348,Arunachal Pradesh,Lohit,8236.0,0.0,0.0,NaN
2868,Arunachal Pradesh,Tirap,12104.0,0.0,0.0,NaN
3284,Arunachal Pradesh,West Siang,14222.0,0.0,0.0,NaN
4116,Assam,Dhubri,4691.0,0.0,0.0,NaN
4740,Assam,Jorhat,8435.0,0.0,0.0,NaN
5052,Assam,Karbi Anglong,6822.0,0.0,0.0,NaN
5572,Assam,Nagaon,5670.0,0.0,0.0,NaN


In [10]:
india_change['nfhs4_zero_status'] = (
    india_change['NFHS 4'] == 0
)

formula_audit = (
    india_change.groupby('Indicator')
    .apply(
        lambda group: pd.Series({
            'Total_records': len(group),
            'NFHS4_zero_records': (
                group['NFHS 4'] == 0
            ).sum(),
            'NFHS4_nonzero_records': (
                group['NFHS 4'] != 0
            ).sum(),
            'Percentage_change_matches': (
                group.loc[
                    group['NFHS 4'] != 0,
                    'change_matches_pct'
                ].sum()
            ),
            'Nonzero_baseline_match_rate': (
                group.loc[
                    group['NFHS 4'] != 0,
                    'change_matches_pct'
                ].mean() * 100
            )
        })
    )
    .reset_index()
)

display(
    formula_audit.sort_values(
        'Nonzero_baseline_match_rate',
        ascending=True
    ).head(20)
)

,Indicator,Total_records,NFHS4_zero_records,NFHS4_nonzero_records,Percentage_change_matches,Nonzero_baseline_match_rate
7,Births in a private health facility that were ...,644.0,193.0,451.0,0.0,0.0
23,Children age 6-59 months who are anaemic,644.0,58.0,586.0,0.0,0.0
26,Children born at home who were taken to a heal...,644.0,514.0,130.0,0.0,0.0
24,Children age 6-8 months receiving solid or sem...,644.0,605.0,39.0,0.0,0.0
31,Children under 5 years who are wasted (weight ...,644.0,58.0,586.0,0.0,0.0
68,Male Moderately or severely elevated blood pre...,644.0,633.0,11.0,0.0,0.0
50,Female Mildly elevated blood pressure (Systoli...,644.0,633.0,11.0,0.0,0.0
51,Female Moderately or severely elevated blood p...,644.0,633.0,11.0,0.0,0.0
44,Ever undergone a screening test for cervical c...,644.0,633.0,11.0,0.0,0.0
45,Ever undergone an oral cavity examination for ...,644.0,633.0,11.0,0.0,0.0


In [11]:
# Recalculate percentage-change matches directly
valid_baseline = india_change['NFHS 4'] != 0

india_change['pct_match_verified'] = False

india_change.loc[valid_baseline, 'pct_match_verified'] = np.isclose(
    india_change.loc[valid_baseline, 'Change'],
    india_change.loc[valid_baseline, 'calculated_pct_change'],
    atol=0.11
)

# Indicator-wise audit
formula_audit_corrected = (
    india_change.groupby('Indicator')
    .apply(
        lambda g: pd.Series({
            'Total_records': len(g),
            'NFHS4_zero_records': (g['NFHS 4'] == 0).sum(),
            'NFHS4_nonzero_records': (g['NFHS 4'] != 0).sum(),
            'Percentage_change_matches': g['pct_match_verified'].sum(),
        })
    )
    .reset_index()
)

formula_audit_corrected['Nonzero_baseline_match_rate'] = (
    formula_audit_corrected['Percentage_change_matches']
    / formula_audit_corrected['NFHS4_nonzero_records'].replace(0, np.nan)
) * 100

display(
    formula_audit_corrected.sort_values(
        'Nonzero_baseline_match_rate',
        ascending=False
    ).head(20)
)

,Indicator,Total_records,NFHS4_zero_records,NFHS4_nonzero_records,Percentage_change_matches,Nonzero_baseline_match_rate
4,Average out-of-pocket expenditure per delivery...,644,61,583,583,100.000000
90,Sex ratio at birth for children born in the la...,644,58,586,586,100.000000
91,Sex ratio of the total population (females per...,644,58,586,586,100.000000
57,Households using iodized salt (%),644,58,586,340,58.020478
84,Population living in households with an improv...,644,58,586,317,54.095563
20,Children age 12-23 months who received most of...,644,77,567,274,48.324515
85,Population living in households with electrici...,644,58,586,275,46.928328
17,Children age 12-23 months who have received BC...,644,74,570,248,43.508772
33,Children under age 5 years whose birth was reg...,644,58,586,184,31.399317
89,Registered pregnancies for which the mother re...,644,58,586,170,29.010239


In [12]:
nutrition_indicators = [
    'Children under 5 years who are stunted (height for age) (%)',
    'Children under 5 years who are wasted (weight for height) (%)',
    'Children under 5 years who are underweight (weight for age) (%)'
]

nutrition_change_audit = formula_audit_corrected[
    formula_audit_corrected['Indicator'].isin(
        nutrition_indicators
    )
].copy()

display(nutrition_change_audit)

,Indicator,Total_records,NFHS4_zero_records,NFHS4_nonzero_records,Percentage_change_matches,Nonzero_baseline_match_rate
29,Children under 5 years who are stunted (height...,644,58,586,2,0.341297
30,Children under 5 years who are underweight (we...,644,58,586,1,0.170648
31,Children under 5 years who are wasted (weight ...,644,58,586,0,0.000000


In [13]:
for indicator in nutrition_indicators:

    subset = india_change[
        india_change['Indicator'] == indicator
    ].copy()

    subset['absolute_difference'] = (
        subset['NFHS 5'] - subset['NFHS 4']
    )

    subset['reverse_difference'] = (
        subset['NFHS 4'] - subset['NFHS 5']
    )

    print("\nINDICATOR:", indicator)

    display(
        subset[
            [
                'State',
                'District Name',
                'NFHS 5',
                'NFHS 4',
                'Change',
                'absolute_difference',
                'reverse_difference'
            ]
        ].head(15)
    )


INDICATOR: Children under 5 years who are stunted (height for age) (%)


,State,District Name,NFHS 5,NFHS 4,Change,absolute_difference,reverse_difference
17,Andaman & Nicobar Island,Nicobar,21.6,20.3,-1.3,1.3,-1.3
121,Andaman & Nicobar Island,North & Middle Andaman,27.0,32.5,5.5,-5.5,5.5
225,Andaman & Nicobar Island,South Andaman,21.1,20.1,-1.0,1.0,-1.0
329,Andhra Pradesh,Anantapur,36.0,40.3,4.3,-4.3,4.3
433,Andhra Pradesh,Chittoor,27.1,31.4,4.3,-4.3,4.3
537,Andhra Pradesh,East Godavari,23.1,27.7,4.6,-4.6,4.6
641,Andhra Pradesh,Guntur,23.8,22.1,-1.7,1.7,-1.7
745,Andhra Pradesh,Krishna,29.8,22.6,-7.2,7.2,-7.2
849,Andhra Pradesh,Kurnool,50.5,44.1,-6.4,6.4,-6.4
953,Andhra Pradesh,Prakasam,22.6,28.2,5.6,-5.6,5.6



INDICATOR: Children under 5 years who are wasted (weight for height) (%)


,State,District Name,NFHS 5,NFHS 4,Change,absolute_difference,reverse_difference
19,Andaman & Nicobar Island,Nicobar,15.7,7.4,-8.3,8.3,-8.3
123,Andaman & Nicobar Island,North & Middle Andaman,27.0,31.7,4.7,-4.7,4.7
227,Andaman & Nicobar Island,South Andaman,12.6,15.6,3.0,-3.0,3.0
331,Andhra Pradesh,Anantapur,19.3,15.2,-4.1,4.1,-4.1
435,Andhra Pradesh,Chittoor,14.8,18.1,3.3,-3.3,3.3
539,Andhra Pradesh,East Godavari,14.3,15.3,1.0,-1.0,1.0
643,Andhra Pradesh,Guntur,17.8,17.9,0.1,-0.1,0.1
747,Andhra Pradesh,Krishna,14.3,20.7,6.4,-6.4,6.4
851,Andhra Pradesh,Kurnool,16.7,18.8,2.1,-2.1,2.1
955,Andhra Pradesh,Prakasam,8.7,15.3,6.6,-6.6,6.6



INDICATOR: Children under 5 years who are underweight (weight for age) (%)


,State,District Name,NFHS 5,NFHS 4,Change,absolute_difference,reverse_difference
18,Andaman & Nicobar Island,Nicobar,24.6,10.5,-14.1,14.1,-14.1
122,Andaman & Nicobar Island,North & Middle Andaman,42.8,33.5,-9.3,9.3,-9.3
226,Andaman & Nicobar Island,South Andaman,17.4,18.5,1.1,-1.1,1.1
330,Andhra Pradesh,Anantapur,40.6,39.3,-1.3,1.3,-1.3
434,Andhra Pradesh,Chittoor,27.9,32.4,4.5,-4.5,4.5
538,Andhra Pradesh,East Godavari,22.4,27.1,4.7,-4.7,4.7
642,Andhra Pradesh,Guntur,26.9,29.1,2.2,-2.2,2.2
746,Andhra Pradesh,Krishna,21.1,27.7,6.6,-6.6,6.6
850,Andhra Pradesh,Kurnool,46.3,37.5,-8.8,8.8,-8.8
954,Andhra Pradesh,Prakasam,24.7,30.3,5.6,-5.6,5.6


In [14]:
for indicator in nutrition_indicators:

    subset = india_change[
        india_change['Indicator'] == indicator
    ].copy()

    expected_change = subset['NFHS 4'] - subset['NFHS 5']

    matches = np.isclose(
        subset['Change'],
        expected_change,
        atol=0.11
    )

    print("\nIndicator:", indicator)
    print("Total records:", len(subset))
    print("Formula matches:", matches.sum())
    print("Formula mismatches:", (~matches).sum())

    if (~matches).sum() > 0:
        display(
            subset.loc[
                ~matches,
                [
                    'State', 'District Name',
                    'NFHS 5', 'NFHS 4', 'Change'
                ]
            ]
        )


Indicator: Children under 5 years who are stunted (height for age) (%)
Total records: 644
Formula matches: 643
Formula mismatches: 1


,State,District Name,NFHS 5,NFHS 4,Change
55657,Telangana,Warangal Urban,29.5,26.6,-9.5



Indicator: Children under 5 years who are wasted (weight for height) (%)
Total records: 644
Formula matches: 643
Formula mismatches: 1


,State,District Name,NFHS 5,NFHS 4,Change
55659,Telangana,Warangal Urban,29.5,16.6,-2.4



Indicator: Children under 5 years who are underweight (weight for age) (%)
Total records: 644
Formula matches: 643
Formula mismatches: 1


,State,District Name,NFHS 5,NFHS 4,Change
55658,Telangana,Warangal Urban,36.0,29.1,-8.8


In [15]:
warangal_check = india_change[
    (india_change['State'] == 'Telangana') &
    (india_change['District Name'] == 'Warangal Urban') &
    (india_change['Indicator'].isin(nutrition_indicators))
].copy()

warangal_check['expected_change'] = (
    warangal_check['NFHS 4'] - warangal_check['NFHS 5']
)

warangal_check['difference_error'] = (
    warangal_check['Change'] -
    warangal_check['expected_change']
)

display(
    warangal_check[
        [
            'District Name',
            'DISTRICT',
            'DT_CEN_CD',
            'Indicator',
            'NFHS 5',
            'NFHS 4',
            'Change',
            'expected_change',
            'difference_error'
        ]
    ]
)

,District Name,DISTRICT,DT_CEN_CD,Indicator,NFHS 5,NFHS 4,Change,expected_change,difference_error
55657,Warangal Urban,Warangal_9,9.0,Children under 5 years who are stunted (height...,29.5,26.6,-9.5,-2.9,-6.6
55658,Warangal Urban,Warangal_9,9.0,Children under 5 years who are underweight (we...,36.0,29.1,-8.8,-6.9,-1.9
55659,Warangal Urban,Warangal_9,9.0,Children under 5 years who are wasted (weight ...,29.5,16.6,-2.4,-12.9,10.5


In [16]:
warangal_india = india_analysis[
    (india_analysis['State_clean'] == 'Telangana') &
    (india_analysis['District'] == 'Warangal Urban') &
    (india_analysis['Indicator'].isin(nutrition_indicators))
]

display(
    warangal_india[
        [
            'State',
            'District',
            'DISTRICT',
            'DT_CEN_CD',
            'Indicator',
            'NFHS 5',
            'NFHS 4'
        ]
    ]
)

NameError: name 'india_analysis' is not defined

In [17]:
import pandas as pd

india = pd.read_csv(
    r"../data/raw/India.csv",
    low_memory=False
)

print("Shape:", india.shape)

Shape: (73632, 9)


In [18]:
warangal_india = india[
    (india['State'].str.strip() == 'Telangana') &
    (india['District'].str.strip() == 'Warangal Urban') &
    (india['Indicator'].str.strip().isin(nutrition_indicators))
]

display(
    warangal_india[
        ['State', 'District', 'DISTRICT',
         'DT_CEN_CD', 'Indicator', 'NFHS 5', 'NFHS 4']
    ]
)

,State,District,DISTRICT,DT_CEN_CD,Indicator,NFHS 5,NFHS 4
61432,Telangana,Warangal Urban,Warangal_9,9,Children under 5 years who are stunted (height...,29.5,0.0
61433,Telangana,Warangal Urban,Warangal_9,9,Children under 5 years who are wasted (weight ...,29.5,0.0
61435,Telangana,Warangal Urban,Warangal_9,9,Children under 5 years who are underweight (we...,36.0,0.0


In [19]:
warangal_india_change = india_change[
    (india_change['State'].str.strip() == 'Telangana') &
    (india_change['District Name'].str.strip() == 'Warangal Urban') &
    (india_change['Indicator'].str.strip().isin(nutrition_indicators))
]

display(
    warangal_india_change[
        ['State', 'District Name', 'Indicator',
         'NFHS 5', 'NFHS 4', 'Change']
    ]
)

,State,District Name,Indicator,NFHS 5,NFHS 4,Change
55657,Telangana,Warangal Urban,Children under 5 years who are stunted (height...,29.5,26.6,-9.5
55658,Telangana,Warangal Urban,Children under 5 years who are underweight (we...,36.0,29.1,-8.8
55659,Telangana,Warangal Urban,Children under 5 years who are wasted (weight ...,29.5,16.6,-2.4


In [21]:
# Compare all three malnutrition indicators
# between India.csv and India_Change.csv

for indicator in nutrition_indicators:

    print("\n" + "=" * 80)
    print("INDICATOR:", indicator)

    a = india[
        (india['State'].str.strip() == 'Telangana') &
        (india['District'].str.strip() == 'Warangal Urban') &
        (india['Indicator'].str.strip() == indicator)
    ]

    b = india_change[
        (india_change['State'].str.strip() == 'Telangana') &
        (india_change['District Name'].str.strip() == 'Warangal Urban') &
        (india_change['Indicator'].str.strip() == indicator)
    ]

    print("\nIndia.csv:")
    display(a[['NFHS 5', 'NFHS 4']])

    print("\nIndia_Change.csv:")
    display(b[['NFHS 5', 'NFHS 4', 'Change']])


INDICATOR: Children under 5 years who are stunted (height for age) (%)

India.csv:


,NFHS 5,NFHS 4
61432,29.5,0.0



India_Change.csv:


,NFHS 5,NFHS 4,Change
55657,29.5,26.6,-9.5



INDICATOR: Children under 5 years who are wasted (weight for height) (%)

India.csv:


,NFHS 5,NFHS 4
61433,29.5,0.0


... (2 outputs left out)


INDICATOR: Children under 5 years who are underweight (weight for age) (%)

India.csv:


,NFHS 5,NFHS 4
61435,36.0,0.0



India_Change.csv:


,NFHS 5,NFHS 4,Change
55658,36.0,29.1,-8.8


In [22]:
# Filter the three malnutrition indicators for Telangana

telangana_india = india[
    (india['State'].str.strip() == 'Telangana') &
    (india['Indicator'].str.strip().isin(nutrition_indicators))
].copy()

telangana_change = india_change[
    (india_change['State'].str.strip() == 'Telangana') &
    (india_change['Indicator'].str.strip().isin(nutrition_indicators))
].copy()

print("India.csv shape:", telangana_india.shape)
print("India_Change.csv shape:", telangana_change.shape)

print("\nIndia.csv NFHS-4 zero values:")
display(
    telangana_india.groupby('Indicator')['NFHS 4']
    .agg(
        total='count',
        zero_values=lambda x: (x == 0).sum(),
        unique_values='nunique'
    )
)

print("\nIndia_Change.csv NFHS-4 zero values:")
display(
    telangana_change.groupby('Indicator')['NFHS 4']
    .agg(
        total='count',
        zero_values=lambda x: (x == 0).sum(),
        unique_values='nunique'
    )
)

India.csv shape: (93, 9)
India_Change.csv shape: (33, 17)

India.csv NFHS-4 zero values:


,total,zero_values,unique_values
Indicator,,,
Children under 5 years who are stunted (height for age) (%),31,31,1
Children under 5 years who are underweight (weight for age) (%),31,31,1
Children under 5 years who are wasted (weight for height) (%),31,31,1



India_Change.csv NFHS-4 zero values:


,total,zero_values,unique_values
Indicator,,,
Children under 5 years who are stunted (height for age) (%),11,0,9
Children under 5 years who are underweight (weight for age) (%),11,0,8
Children under 5 years who are wasted (weight for height) (%),11,0,7


In [23]:
# Get unique Telangana district names from both datasets

india_districts = set(
    india.loc[
        india['State'].str.strip() == 'Telangana',
        'District'
    ].dropna().str.strip().unique()
)

change_districts = set(
    india_change.loc[
        india_change['State'].str.strip() == 'Telangana',
        'District Name'
    ].dropna().str.strip().unique()
)

print("Districts in India.csv:", len(india_districts))
print("Districts in India_Change.csv:", len(change_districts))

print("\nDistricts present in BOTH:")
print(sorted(india_districts & change_districts))

print("\nDistricts ONLY in India.csv:")
print(sorted(india_districts - change_districts))

print("\nDistricts ONLY in India_Change.csv:")
print(sorted(change_districts - india_districts))

Districts in India.csv: 31
Districts in India_Change.csv: 11

Districts present in BOTH:
['Adilabad', 'Hyderabad', 'Karimnagar', 'Khammam', 'Mahabubnagar', 'Medak', 'Nalgonda', 'Nizamabad', 'Ranga Reddy', 'Warangal Rural', 'Warangal Urban']

Districts ONLY in India.csv:
['Bhadradri Kothagudem', 'Jagitial', 'Jangoan', 'Jayashankar Bhupalapally', 'Jogulamba Gadwal', 'Kamareddy', 'Komaram Bheem Asifabad', 'Mahabubabad', 'Mancherial', 'MedchalMalkajgiri', 'Nagarkurnool', 'Nirmal', 'Peddapalli', 'Rajanna Sircilla', 'Sangareddy', 'Siddipet', 'Suryapet', 'Vikarabad', 'Wanaparthy', 'Yadadri Bhuvanagiri']

Districts ONLY in India_Change.csv:
[]


In [24]:
telangana_comparison = telangana_change[
    telangana_change['Indicator'].str.strip().isin(nutrition_indicators)
].copy()

display(
    telangana_comparison[
        [
            'State',
            'District Name',
            'DISTRICT',
            'DT_CEN_CD',
            'Indicator',
            'NFHS 5',
            'NFHS 4',
            'Change'
        ]
    ].sort_values(['District Name', 'Indicator'])
)

,State,District Name,DISTRICT,DT_CEN_CD,Indicator,NFHS 5,NFHS 4,Change
54617,Telangana,Adilabad,Adilabad_1,1.0,Children under 5 years who are stunted (height...,45.7,38.0,-7.7
54618,Telangana,Adilabad,Adilabad_1,1.0,Children under 5 years who are underweight (we...,52.0,36.0,-16.0
54619,Telangana,Adilabad,Adilabad_1,1.0,Children under 5 years who are wasted (weight ...,29.5,22.0,-7.5
54721,Telangana,Hyderabad,Hyderabad_5,5.0,Children under 5 years who are stunted (height...,30.4,16.0,-14.4
54722,Telangana,Hyderabad,Hyderabad_5,5.0,Children under 5 years who are underweight (we...,18.9,17.0,-1.9
54723,Telangana,Hyderabad,Hyderabad_5,5.0,Children under 5 years who are wasted (weight ...,17.7,15.0,-2.7
54825,Telangana,Karimnagar,Karimnagar_3,3.0,Children under 5 years who are stunted (height...,30.3,24.0,-6.3
54826,Telangana,Karimnagar,Karimnagar_3,3.0,Children under 5 years who are underweight (we...,27.5,25.0,-2.5
54827,Telangana,Karimnagar,Karimnagar_3,3.0,Children under 5 years who are wasted (weight ...,13.9,19.0,5.1
54929,Telangana,Khammam,Khammam_10,10.0,Children under 5 years who are stunted (height...,22.2,27.0,4.8


In [25]:
import numpy as np

# Work on a copy
telangana_formula_audit = telangana_comparison.copy()

# Calculate expected change
telangana_formula_audit['Expected_Change'] = (
    telangana_formula_audit['NFHS 4'] -
    telangana_formula_audit['NFHS 5']
)

# Calculate difference between stored and expected change
telangana_formula_audit['Change_Error'] = (
    telangana_formula_audit['Change'] -
    telangana_formula_audit['Expected_Change']
)

# Check whether stored change matches expected change
telangana_formula_audit['Formula_Match'] = np.isclose(
    telangana_formula_audit['Change'],
    telangana_formula_audit['Expected_Change'],
    atol=0.11
)

# Display mismatches
formula_mismatches = telangana_formula_audit[
    ~telangana_formula_audit['Formula_Match']
]

print("Total records:", len(telangana_formula_audit))
print("Formula matches:", telangana_formula_audit['Formula_Match'].sum())
print("Formula mismatches:", len(formula_mismatches))

display(
    formula_mismatches[
        [
            'District Name',
            'Indicator',
            'NFHS 5',
            'NFHS 4',
            'Change',
            'Expected_Change',
            'Change_Error'
        ]
    ]
)

Total records: 33
Formula matches: 30
Formula mismatches: 3


,District Name,Indicator,NFHS 5,NFHS 4,Change,Expected_Change,Change_Error
55657,Warangal Urban,Children under 5 years who are stunted (height...,29.5,26.6,-9.5,-2.9,-6.6
55658,Warangal Urban,Children under 5 years who are underweight (we...,36.0,29.1,-8.8,-6.9,-1.9
55659,Warangal Urban,Children under 5 years who are wasted (weight ...,29.5,16.6,-2.4,-12.9,10.5


In [26]:
telangana_issue_log = formula_mismatches[
    [
        'District Name',
        'Indicator',
        'NFHS 5',
        'NFHS 4',
        'Change',
        'Expected_Change',
        'Change_Error'
    ]
].copy()

telangana_issue_log['Issue'] = (
    'Stored Change does not equal NFHS 4 minus NFHS 5'
)

telangana_issue_log['Status'] = 'Requires source verification'

display(telangana_issue_log)

,District Name,Indicator,NFHS 5,NFHS 4,Change,Expected_Change,Change_Error,Issue,Status
55657,Warangal Urban,Children under 5 years who are stunted (height...,29.5,26.6,-9.5,-2.9,-6.6,Stored Change does not equal NFHS 4 minus NFHS 5,Requires source verification
55658,Warangal Urban,Children under 5 years who are underweight (we...,36.0,29.1,-8.8,-6.9,-1.9,Stored Change does not equal NFHS 4 minus NFHS 5,Requires source verification
55659,Warangal Urban,Children under 5 years who are wasted (weight ...,29.5,16.6,-2.4,-12.9,10.5,Stored Change does not equal NFHS 4 minus NFHS 5,Requires source verification


In [27]:
print("Total records:", len(india_change_audit))
print("Formula matches:", india_change_audit['Formula_Match'].sum())
print("Formula mismatches:", (~india_change_audit['Formula_Match']).sum())

print(
    "Mismatch percentage:",
    round((~india_change_audit['Formula_Match']).mean() * 100, 2),
    "%"
)

NameError: name 'india_change_audit' is not defined

In [28]:
import numpy as np
import pandas as pd

india_change_audit = india_change.copy()

india_change_audit['Expected_Change'] = (
    india_change_audit['NFHS 4'] -
    india_change_audit['NFHS 5']
)

india_change_audit['Formula_Match'] = np.isclose(
    india_change_audit['Change'],
    india_change_audit['Expected_Change'],
    atol=0.11
)

print("Audit dataframe created:", india_change_audit.shape)

Audit dataframe created: (66976, 19)


In [29]:
total = len(india_change_audit)
matches = india_change_audit['Formula_Match'].sum()
mismatches = total - matches

print("Total records:", total)
print("Formula matches:", matches)
print("Formula mismatches:", mismatches)
print("Mismatch percentage:", round(mismatches / total * 100, 2), "%")

Total records: 66976
Formula matches: 16505
Formula mismatches: 50471
Mismatch percentage: 75.36 %


In [30]:
indicator_formula_audit = (
    india_change_audit.groupby('Indicator')
    .agg(
        total_records=('Formula_Match', 'size'),
        matches=('Formula_Match', 'sum'),
        mismatches=('Formula_Match', lambda x: (~x).sum())
    )
    .reset_index()
)

indicator_formula_audit['mismatch_pct'] = (
    indicator_formula_audit['mismatches'] /
    indicator_formula_audit['total_records'] * 100
)

display(
    indicator_formula_audit.sort_values(
        'mismatches',
        ascending=False
    ).head(20)
)

,Indicator,total_records,matches,mismatches,mismatch_pct
2,Any method (%),644,1,643,99.84472
3,Any modern method (%),644,1,643,99.84472
35,Children who received postnatal care from a do...,644,1,643,99.84472
51,Female Moderately or severely elevated blood p...,644,1,643,99.84472
56,Households using clean fuel for cooking (%),644,1,643,99.84472
49,Female Elevated blood pressure or taking medic...,644,1,643,99.84472
46,Female Blood sugar level high (141-160 mg/dl)...,644,1,643,99.84472
48,Female Blood sugar level very high (>160 mg/d...,644,1,643,99.84472
47,Female Blood sugar level high or very high (>...,644,1,643,99.84472
74,Mothers who consumed iron folic acid for 180 d...,644,1,643,99.84472


In [31]:
import numpy as np

audit = india_change.copy()

# Formula 1: NFHS 4 - NFHS 5
audit['F1'] = audit['NFHS 4'] - audit['NFHS 5']

# Formula 2: NFHS 5 - NFHS 4
audit['F2'] = audit['NFHS 5'] - audit['NFHS 4']

# Formula 3: Percentage change relative to NFHS 4
audit['F3'] = np.where(
    audit['NFHS 4'] != 0,
    (audit['NFHS 5'] - audit['NFHS 4']) /
    audit['NFHS 4'] * 100,
    np.nan
)

# Formula 4: Percentage change, reverse direction
audit['F4'] = np.where(
    audit['NFHS 4'] != 0,
    (audit['NFHS 4'] - audit['NFHS 5']) /
    audit['NFHS 4'] * 100,
    np.nan
)

# Compare stored Change with each formula
for col in ['F1', 'F2', 'F3', 'F4']:
    audit[col + '_match'] = np.isclose(
        audit['Change'],
        audit[col],
        atol=0.11,
        equal_nan=False
    )

# Overall formula comparison
for col in ['F1', 'F2', 'F3', 'F4']:
    print(
        col,
        "matches:",
        audit[col + '_match'].sum()
    )

F1 matches: 16505
F2 matches: 54301
F3 matches: 4536
F4 matches: 468
